# Трансформеры — базовые и дополнительные эксперименты

Исходные Colab (копии оригиналов):
- [Трансформеры.ipynb](https://colab.research.google.com/github/SamDerden/SpeechSense/blob/main/%D0%A2%D1%80%D0%B0%D0%BD%D1%81%D1%84%D0%BE%D1%80%D0%BC%D0%B5%D1%80%D1%8B.ipynb)
- [ru_sentiment_research_FINAL.ipynb](https://colab.research.google.com/github/SamDerden/SpeechSense/blob/main/ru_sentiment_research_FINAL.ipynb)

Перенос исходного кода; выполнение и метрики пока не проверены. Исторические результаты в оригиналах.

In [ ]:
from pathlib import Path
import os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'download_dataset.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Clone SpeechSense and run from its root or notebooks directory')
os.chdir(ROOT)
DATA_PATH = ROOT / 'data' / 'merged_sentiment_dataset.csv.gz'
if not DATA_PATH.exists():
    raise FileNotFoundError('Run python data/download_dataset.py first')


In [ ]:
!pip install imbalanced-learn optuna plotly -q
!pip install -q transformers datasets accelerate torchmetrics
!pip install -q sentencepiece  # нужен для некоторых токенизаторов
!pip install -q nlpaug  # для контекстной аугментации
!pip install -q sacremoses  # для back-translation

import os
import re
import gc
import random
import warnings
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup
)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Фиксация seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Маппинг классов
CLASS_MAP = {0: "Нейтральный", 1: "Позитивный", 2: "Негативный"}
CLASS_NAMES = [CLASS_MAP[i] for i in range(3)]
NUM_CLASSES = 3

print(" Все библиотеки импортированы!")

In [ ]:
import pandas as pd
df = pd.read_csv(DATA_PATH, compression='gzip')

In [ ]:
print("ОЧИСТКА ТЕКСТА")

def clean_text(text: str) -> str:
    """Базовая очистка текста для трансформеров (минимальная!)."""
    if not isinstance(text, str):
        return ""
    text = text[:10000]  # обрезаем экстремально длинные
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[@#]\w+', ' ', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r'[^а-яёa-z0-9\s.!?,-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print(f"Размер ДО очистки: {df.shape[0]:,}")
df_clean = df.copy()
df_clean['text_clean'] = df_clean['text'].astype(str).apply(clean_text)

# Фильтр коротких текстов
MIN_LEN = 10
df_clean = df_clean[df_clean['text_clean'].str.len() >= MIN_LEN].copy()
print(f"Размер ПОСЛЕ очистки: {df_clean.shape[0]:,}")

# Распределение после очистки
print("\n=== Распределение классов в очищенном датасете ===")
print(df_clean['sentiment'].value_counts().sort_index())
print("\nДоли (%):")
print((df_clean['sentiment'].value_counts(normalize=True).sort_index() * 100).round(2))


In [ ]:
print("РАЗДЕЛЕНИЕ ДАННЫХ (70/15/15)")

from sklearn.model_selection import train_test_split

# Стратифицированное разделение 70/15/15
train_df, temp_df = train_test_split(
    df_clean, test_size=0.3, random_state=SEED,
    stratify=df_clean['sentiment']
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=SEED,
    stratify=temp_df['sentiment']
)

print(f" Train: {len(train_df):,} ({len(train_df)/len(df_clean)*100:.1f}%)")
print(f" Validation: {len(val_df):,} ({len(val_df)/len(df_clean)*100:.1f}%)")
print(f" Test: {len(test_df):,} ({len(test_df)/len(df_clean)*100:.1f}%)")

print(f"\n Распределение в train:")
print(train_df['sentiment'].value_counts().sort_index())
print(f"\n Распределение в validation:")
print(val_df['sentiment'].value_counts().sort_index())
print(f"\n Распределение в test:")
print(test_df['sentiment'].value_counts().sort_index())

# Трансформеры

In [ ]:
print("УНИВЕРСАЛЬНАЯ ФУНКЦИЯ ОБУЧЕНИЯ")

import os
from typing import Dict, List, Optional, Tuple

# =============================================================================
# 1. Кастомный Dataset для трансформеров
# =============================================================================
class SentimentDataset(Dataset):
    """
    Кастомный Dataset для трансформеров.
    Превращает текст в токены, добавляет padding и attention mask.
    """
    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_length: int = 128):
        self.texts = [str(t) for t in texts]
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Токенизация текста
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

print(" Класс SentimentDataset определён")

# =============================================================================
# 2. Функция оценки модели
# =============================================================================
def evaluate_model(model, data_loader, device):
    """Оценка модели на validation/test."""
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for batch in data_loader:
            out = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device)
            )
            preds.extend(torch.argmax(out.logits, dim=-1).cpu().numpy())
            labels.extend(batch['labels'].cpu().numpy())

    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    accuracy = accuracy_score(labels, preds)
    return macro_f1, accuracy

print(" Функция evaluate_model определена")

# =============================================================================
# 3. Универсальная функция обучения трансформера
# =============================================================================
def train_transformer(
    model_name: str,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    text_col: str = 'text',
    label_col: str = 'sentiment',
    max_length: int = 128,
    batch_size: int = 16,
    learning_rate: float = 2e-5,
    epochs: int = 3,
    use_differential_lr: bool = False,
    label_smoothing: float = 0.0,
    gradient_clip_norm: float = 1.0,
    custom_loss_fn: Optional[nn.Module] = None,  # ← НОВЫЙ ПАРАМЕТР
    model_key: str = "",
    save_dir: Optional[str] = None,
) -> Dict:
    """
    Универсальная функция обучения трансформера.

    Args:
        model_name: название модели (например, 'DeepPavlov/rubert-base-cased')
        train_df: обучающая выборка
        val_df: валидационная выборка (НЕ тест!)
        text_col: колонка с текстом
        label_col: колонка с метками
        max_length: максимальная длина последовательности
        batch_size: размер батча
        learning_rate: скорость обучения
        epochs: количество эпох
        use_differential_lr: использовать дифференциальные LR
        label_smoothing: коэффициент label smoothing
        gradient_clip_norm: норма для gradient clipping
        custom_loss_fn: кастомная функция потерь (например, с весами классов)
        model_key: ключ модели для логирования
        save_dir: директория для сохранения модели

    Returns:
        Dict с результатами обучения
    """
    print(f"   Модель: {model_name}")
    print(f"   Train: {len(train_df)}, Val: {len(val_df)}")
    print(f"   Batch size: {batch_size}, LR: {learning_rate}, Epochs: {epochs}")
    print(f"   Max length: {max_length}")

    # 1. Загрузка токенизатора и модели
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=3
    ).to(DEVICE)

    # 2. Создание DataLoader (train + val)
    train_dataset = SentimentDataset(
        train_df[text_col].tolist(),
        train_df[label_col].tolist(),
        tokenizer, max_length,
    )
    val_dataset = SentimentDataset(
        val_df[text_col].tolist(),
        val_df[label_col].tolist(),
        tokenizer, max_length,
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    print(f" Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

    # 3. Оптимизатор
    if use_differential_lr:
        no_decay = ['bias', 'LayerNorm.weight']
        optimizer_grouped = [
            {'params': [p for n, p in model.bert.named_parameters()
                        if not any(nd in n for nd in no_decay)
                        and any(f'layer.{i}.' in n for i in range(6))],
             'weight_decay': 0.01, 'lr': learning_rate * 0.1},
            {'params': [p for n, p in model.bert.named_parameters()
                        if not any(nd in n for nd in no_decay)
                        and any(f'layer.{i}.' in n for i in range(6, 12))],
             'weight_decay': 0.01, 'lr': learning_rate},
            {'params': model.classifier.parameters(),
             'weight_decay': 0.01, 'lr': learning_rate * 5},
            {'params': [p for n, p in model.named_parameters()
                        if any(nd in n for nd in no_decay)],
             'weight_decay': 0.0, 'lr': learning_rate},
        ]
        optimizer = torch.optim.AdamW(optimizer_grouped)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

    # Scheduler (Cosine Annealing with Warmup)
    total_steps = len(train_loader) * epochs
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * 0.1),
        num_training_steps=total_steps,
    )

    # Функция потерь
    if custom_loss_fn is not None:
        loss_fn = custom_loss_fn
        print(f"    Используем кастомную функцию потерь: {type(loss_fn).__name__}")
    else:
        loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        print(f"    Используем CrossEntropyLoss (label_smoothing={label_smoothing})")

    # 4. Цикл обучения с оценкой на VALIDATION
    print(f" Начинаем обучение...")
    best_val_f1 = 0.0
    best_model_state = None

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            # Используем кастомную или стандартную функцию потерь
            loss = loss_fn(outputs.logits, labels)

            loss.backward()

            # Gradient clipping
            if gradient_clip_norm > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip_norm)

            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        # Оценка на VALIDATION после каждой эпохи
        val_f1, val_acc = evaluate_model(model, val_loader, DEVICE)
        avg_loss = total_loss / len(train_loader)
        print(f"   Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}")

        # Сохраняем лучшую модель по validation F1
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}

    # Загружаем лучшую модель
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    # Сохранение модели
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print(f"    Сохранено в {save_dir}")

    print(f"\n Лучший Val Macro-F1: {best_val_f1:.4f}")

    return {
        'val_macro_f1': best_val_f1,
        'model': model,
        'tokenizer': tokenizer,
    }

## Определение 5 трансформеров для сравнения

### Почему именно эти модели?
rubert-base-cased — это золотой стандарт для русского NLP. Обучен на огромном корпусе русских текстов (Википедия, новости, форумы). Понимает русскую морфологию, контекст, иронию.

ruRoberta-large — улучшенная версия RuBERT с большим размером модели и более агрессивным маскированием при предобучении. Даёт +2-4% к качеству, но обучается в 3 раза дольше.

rubert-tiny2 — крошечная модель для быстрых экспериментов. Если нужно проверить, что пайплайн работает, — запускай её.

rubert-base-cased-sentiment-rusentiment — уже дообучена на датасете rusentiment (русские отзывы). Можно использовать как отправную точку — её веса уже "настроены" на задачу тональности.

xlm-roberta-base — мультиязычная модель, обученная на 100 языках одновременно. Интересна тем, что показывает, насколько важно использовать именно русскоязычную модель vs мультиязычную.

### Rubert-tiny2

In [ ]:
result_8 = train_transformer(
    model_name='cointegrated/rubert-tiny2',
    train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=64,
    learning_rate=3e-5, epochs=3, model_key='rubert-tiny2',
)
transformer_results = {'rubert-tiny2': result_8}

### Rubert-base

In [ ]:
result_9 = train_transformer(
    model_name='DeepPavlov/rubert-base-cased',
    train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=3, model_key='rubert-base',
)
transformer_results['rubert-base'] = result_9

### Rubert-sentiment

In [ ]:
result_10 = train_transformer(
    model_name='blanchefort/rubert-base-cased-sentiment-rusentiment',
    train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=3, model_key='rubert-sentiment',
)
transformer_results['rubert-sentiment'] = result_10

### Xlm-roberta-base

In [ ]:
result_11 = train_transformer(
    model_name='FacebookAI/xlm-roberta-base',
    train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=3, model_key='xlm-roberta-base',
)
transformer_results['xlm-roberta-base'] = result_11

### RuRoberta-large

In [ ]:
result_12 = train_transformer(
    model_name='sberbank-ai/ruRoberta-large',
    train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=8,
    learning_rate=1e-5, epochs=2, model_key='ruRoberta-large',
)
transformer_results['ruRoberta-large'] = result_12

### СВОДНАЯ ТАБЛИЦА ТРАНСФОРМЕРОВ
> Добавить блок с цитатой



In [ ]:
print("ЯЧЕЙКА 13: СВОДНАЯ ТАБЛИЦА ТРАНСФОРМЕРОВ")

print(f"\n{'Модель':<25} | {'Val Macro-F1':<12}")
print("-" * 40)
for key, res in sorted(transformer_results.items(), key=lambda x: x[1]['val_macro_f1'], reverse=True):
    print(f"{key:<25} | {res['val_macro_f1']:.4f}")

best_tf_key = max(transformer_results, key=lambda k: transformer_results[k]['val_macro_f1'])
best_tf_f1 = transformer_results[best_tf_key]['val_macro_f1']
best_tf_model = transformer_results[best_tf_key]['model']
best_tf_tokenizer = transformer_results[best_tf_key]['tokenizer']

print(f"\n Лучший: {best_tf_key} (Val F1={best_tf_f1:.4f})")

# Визуализация
plt.figure(figsize=(12, 6))
models = list(transformer_results.keys())
f1_values = [transformer_results[k]['val_macro_f1'] for k in models]
plt.bar(models, f1_values, color=['#2ecc71' if v == max(f1_values) else '#3498db' for v in f1_values])
plt.axhline(y=best_tf_f1, color='red', linestyle='--', linewidth=2, label=f'Лучший: {best_tf_f1:.4f}')
plt.xlabel('Модель'), plt.ylabel('Val Macro-F1')
plt.title('Сравнение 5 трансформеров', fontsize=14)
plt.xticks(rotation=45), plt.legend()
plt.tight_layout()
plt.show()

## Аугментация

### Аугментация нейтрального класса - Back-Translation

In [ ]:
print("BACK-TRANSLATION")

from transformers import pipeline

print(" Загрузка моделей перевода...")
translator_ru_en = pipeline('translation', model='Helsinki-NLP/opus-mt-ru-en', device=0 if torch.cuda.is_available() else -1)
translator_en_ru = pipeline('translation', model='Helsinki-NLP/opus-mt-en-ru', device=0 if torch.cuda.is_available() else -1)

def back_translate(text: str) -> str:
    try:
        en = translator_ru_en(text, max_length=128)[0]['translation_text']
        return translator_en_ru(en, max_length=128)[0]['translation_text']
    except:
        return text

# Аугментация только нейтрального класса (label=0)
neutral_mask = train_df['sentiment'] == 0
neutral_texts = train_df.loc[neutral_mask, text_col].tolist()
print(f"Исходных нейтральных: {len(neutral_texts)}")

# Создаём 2 парафраза для каждого
aug_texts, aug_labels = [], []
for i, text in enumerate(neutral_texts[:5000]):  # ограничим для скорости
    if i % 500 == 0:
        print(f"   {i}/{min(5000, len(neutral_texts))}")
    for _ in range(2):
        aug_texts.append(back_translate(text))
        aug_labels.append(0)

# Объединяем с исходными данными
augmented_train_df = pd.concat([
    train_df,
    pd.DataFrame({text_col: aug_texts, 'sentiment': aug_labels})
], ignore_index=True)

print(f"\n Аугментация завершена")
print(f"   Было: {len(train_df):,}")
print(f"   Стало: {len(augmented_train_df):,}")
print(f"   Прирост нейтральных: {len(aug_texts)}")

### Аугментация нейтрального класса - Контекстная замена

In [ ]:
print("КОНТЕКСТНАЯ ЗАМЕНА")

!pip install -q nlpaug

import nlpaug.augmenter.word as naw

context_aug = naw.ContextualWordEmbsAug(
    model_path='distilbert-base-russian-cased',
    action='substitute', aug_min=1, aug_max=3, aug_p=0.2,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

print(" Аугментация нейтрального класса...")
context_aug_texts = []
for i, text in enumerate(neutral_texts[:3000]):
    if i % 500 == 0:
        print(f"   {i}/3000")
    try:
        context_aug_texts.append(context_aug.augment(text))
    except:
        context_aug_texts.append(text)

# Добавляем к аугментированному датасету
augmented_train_df = pd.concat([
    augmented_train_df,
    pd.DataFrame({text_col: context_aug_texts, 'sentiment': [0] * len(context_aug_texts)})
], ignore_index=True)

print(f"\n Финальный размер: {len(augmented_train_df):,}")

## Вычисление весов классов для аугментированного датасета

In [ ]:
print("ВЕСА КЛАССОВ")

class_counts = augmented_train_df['sentiment'].value_counts().sort_index().values
total = len(augmented_train_df)

print(f"\n Распределение:")
for cls, count in enumerate(class_counts):
    print(f"   Класс {cls} ({CLASS_NAMES[cls]}): {count:,} ({count/total*100:.1f}%)")

class_weights_np = total / (NUM_CLASSES * class_counts)
class_weights_tensor = torch.tensor(class_weights_np, dtype=torch.float32).to(DEVICE)

print(f"\n Веса: {class_weights_np.round(4)}")

## Переобучение топ-3 моделей на аугментированных данных с весами

In [ ]:
print("ПЕРЕОБУЧЕНИЕ ТОП-3 НА АУГМЕНТИРОВАННЫХ ДАННЫХ")

# Топ-3 модели по Val F1 из Ячейки 13
top_3_keys = sorted(transformer_results, key=lambda k: transformer_results[k]['val_macro_f1'], reverse=True)[:3]
print(f" Топ-3 модели: {top_3_keys}")

# =============================================================================
# Кастомная функция потерь с весами классов
# =============================================================================
class WeightedCELoss(nn.Module):
    """
    CrossEntropyLoss с весами классов для борьбы с дисбалансом.
    """
    def __init__(self, class_weights: torch.Tensor):
        super().__init__()
        self.weights = class_weights.to(DEVICE)

    def forward(self, logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        return F.cross_entropy(logits, labels, weight=self.weights)

# Создаём функцию потерь с весами из Ячейки 16
weighted_loss_fn = WeightedCELoss(class_weights_tensor)

print(f"\n⚖️ Используем WeightedCELoss с весами: {class_weights_np.round(4)}")

# =============================================================================
# Переобучение каждой из топ-3 моделей
# =============================================================================
augmented_results = {}

for key in top_3_keys:
    # Получаем название модели из tokenizer
    model_name = transformer_results[key]['tokenizer'].name_or_path

    print(f"\n{'=' * 60}")
    print(f"🔄 Переобучение: {key}")
    print(f"{'=' * 60}")

    # Вызываем train_transformer с кастомной функцией потерь
    result = train_transformer(
        model_name=model_name,
        train_df=augmented_train_df,  # ← АУГМЕНТИРОВАННЫЕ данные
        val_df=val_df,                # ← Validation (не test!)
        text_col=text_col,
        max_length=128,
        batch_size=16,
        learning_rate=2e-5,
        epochs=2,
        custom_loss_fn=weighted_loss_fn,  # ← ВЕСА КЛАССОВ
        model_key=f'Augmented-{key}',
    )

    augmented_results[key] = result

# =============================================================================
# Сводная таблица результатов
# =============================================================================
print(" РЕЗУЛЬТАТЫ ПЕРЕОБУЧЕНИЯ")

print(f"\n{'Модель':<30} | {'Val Macro-F1 (до)':<18} | {'Val Macro-F1 (после)':<20} | {'Прирост':<10}")
print("-" * 85)

for key in top_3_keys:
    before_f1 = transformer_results[key]['val_macro_f1']
    after_f1 = augmented_results[key]['val_macro_f1']
    improvement = after_f1 - before_f1

    print(f"{key:<30} | {before_f1:<18.4f} | {after_f1:<20.4f} | {improvement:+.4f}")

# Лучшая модель после аугментации
best_aug_key = max(augmented_results, key=lambda k: augmented_results[k]['val_macro_f1'])
best_aug_f1 = augmented_results[best_aug_key]['val_macro_f1']

print(f"\n Лучшая модель после аугментации: {best_aug_key}")
print(f"   Val Macro-F1: {best_aug_f1:.4f}")

# Сохранение лучшей модели
os.makedirs('./augmented_models', exist_ok=True)
save_path = f'./augmented_models/best_augmented_{best_aug_key}'
augmented_results[best_aug_key]['model'].save_pretrained(save_path)
augmented_results[best_aug_key]['tokenizer'].save_pretrained(save_path)
print(f"    Сохранено в {save_path}")


## Cosine Annealing with Warmup - детальное объяснение и визуализация

In [ ]:
# =============================================================================
# ЯЧЕЙКА 18: COSINE ANNEALING С WARMUP - ВИЗУАЛИЗАЦИЯ
# =============================================================================
print("\n" + "=" * 70)
print("ЯЧЕЙКА 18: COSINE ANNEALING С WARMUP")
print("=" * 70)

print("""
 Cosine Annealing with Warmup:

Этап 1: WARMUP (первые 10% шагов)
  • LR плавно растёт от 0 до целевого значения
  • Зачем: в начале обучения градиенты нестабильны

Этап 2: COSINE ANNEALING (остальные 90%)
  • LR плавно уменьшается по косинусоиде до почти 0
  • Зачем: модель аккуратно скатывается в минимум loss

Почему помогает для Нейтрального класса?
  Нейтраль — редкий класс. Без warmup модель "забывает" его
  в начале обучения. Cosine annealing в конце позволяет
  точнее настроить границу между классами.
""")

# Визуализация
LR_MAX, LR_MIN, WARMUP_RATIO = 2e-5, 1e-6, 0.1
TOTAL_STEPS = 1000
warmup_steps = int(TOTAL_STEPS * WARMUP_RATIO)

lrs = []
for step in range(TOTAL_STEPS):
    if step < warmup_steps:
        lr = LR_MAX * (step / warmup_steps)
    else:
        progress = (step - warmup_steps) / (TOTAL_STEPS - warmup_steps)
        lr = LR_MIN + 0.5 * (LR_MAX - LR_MIN) * (1 + np.cos(np.pi * progress))
    lrs.append(lr)

plt.figure(figsize=(14, 6))
plt.plot(lrs, linewidth=3, color='#3498db')
plt.axvline(x=warmup_steps, color='#e74c3c', linestyle='--', linewidth=2, label=f'Warmup end')
plt.axhline(y=LR_MAX, color='#2ecc71', linestyle=':', label=f'LR max')
plt.xlabel('Training Step', fontsize=12), plt.ylabel('Learning Rate', fontsize=12)
plt.title('Cosine Annealing with Warmup', fontsize=14, fontweight='bold')
plt.legend(), plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n В нашем коде это реализовано через:")
print(f"   get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=..., num_training_steps=...)")

## Ансамблирование топ-3 трансформеров

In [ ]:
print("АНСАМБЛЬ ТОП-3")

# Собираем топ-3 модели (используем augmented_results если есть, иначе transformer_results)
models_to_ensemble = augmented_results if augmented_results else {k: transformer_results[k] for k in top_3_keys}

# Оценка на validation
all_probs, all_labels = [], []
val_dataset_ens = SentimentDataset(
    val_df[text_col].tolist(), val_df['sentiment'].tolist(),
    best_tf_tokenizer, 128
)
val_loader_ens = DataLoader(val_dataset_ens, batch_size=16, shuffle=False)

with torch.no_grad():
    for batch in val_loader_ens:
        probs_batch = []
        for key, res in models_to_ensemble.items():
            model = res['model']
            model.eval()
            out = model(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE)
            )
            probs_batch.append(F.softmax(out.logits, dim=-1).cpu().numpy())
        all_probs.append(np.mean(probs_batch, axis=0))
        all_labels.extend(batch['labels'].cpu().numpy())

all_probs = np.vstack(all_probs)
ensemble_preds = np.argmax(all_probs, axis=1)
ensemble_f1 = f1_score(all_labels, ensemble_preds, average='macro', zero_division=0)

print(f"\n РЕЗУЛЬТАТ АНСАМБЛЯ:")
print(f"   Val Macro-F1: {ensemble_f1:.4f}")

# Сравнение с одиночными моделями
print(f"\n Сравнение:")
for key, res in models_to_ensemble.items():
    print(f"   {key}: {res['val_macro_f1']:.4f}")
print(f"   Ансамбль: {ensemble_f1:.4f} ({ensemble_f1 - max(r['val_macro_f1'] for r in models_to_ensemble.values()):+.4f})")

## Финальный анализ ошибок и сохранение моделей

In [ ]:
print("ФИНАЛЬНЫЙ АНАЛИЗ")

# Определяем лучшую модель (ансамбль или одиночная)
best_overall_f1 = max(ensemble_f1, max(r['val_macro_f1'] for r in models_to_ensemble.values()))

if ensemble_f1 >= max(r['val_macro_f1'] for r in models_to_ensemble.values()):
    print(f" Лучшая: АНСАМБЛЬ (Val F1={ensemble_f1:.4f})")
    best_model_type = 'ensemble'
else:
    best_key = max(models_to_ensemble, key=lambda k: models_to_ensemble[k]['val_macro_f1'])
    print(f" Лучшая: {best_key} (Val F1={models_to_ensemble[best_key]['val_macro_f1']:.4f})")
    best_model_type = 'single'

# Сохранение
os.makedirs('./final_models', exist_ok=True)
if best_model_type == 'ensemble':
    for key, res in models_to_ensemble.items():
        save_path = f'./final_models/ensemble_{key}'
        res['model'].save_pretrained(save_path)
        res['tokenizer'].save_pretrained(save_path)
        print(f"   Сохранён ансамбль {key}")
else:
    save_path = f'./final_models/best_{best_key}'
    models_to_ensemble[best_key]['model'].save_pretrained(save_path)
    models_to_ensemble[best_key]['tokenizer'].save_pretrained(save_path)
    print(f"   Сохранена модель {best_key}")

# Метрика для сравнения в Ячейке 35
baseline_final_f1 = best_overall_f1
print(f"\n Базовая метрика для Ячейки 35: {baseline_final_f1:.4f}")

# Дополнительные эксперименты

## Продвинутые техники обучения

In [ ]:
BEST_MODEL_NAME = best_tf_tokenizer.name_or_path  # или transformer_results[best_tf_key]['tokenizer'].name_or_path
technique_results = {}

### Differential Learning Rates (Разные скорости обучения для слоёв)

Суть: Замороженные слои BERT (нижние) обучаются медленнее, а классификационная голова (верхние слои) — быстрее.

Нижние слои BERT уже выучили общую лингвистику, их не нужно сильно менять. Верхние слои нужно адаптировать под конкретную задачу.

In [ ]:
result_21 = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=2,
    use_differential_lr=True, model_key='21. Differential LR',
)
technique_results['21. Differential LR'] = result_21['val_macro_f1']

### Label Smoothing (Сглаживание меток)

Суть: Вместо жёстких меток [0, 1, 0] используем мягкие [0.1, 0.8, 0.1]. Это не даёт модели быть слишком уверенной и улучшает обобщение.

Модель становится менее "уверенной" в своих предсказаниях, что особенно полезно для размытого класса "Нейтраль".

In [ ]:
result_22 = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=2,
    label_smoothing=0.1, model_key='22. Label Smoothing',
)
technique_results['22. Label Smoothing'] = result_22['val_macro_f1']

### Gradient Clipping (Обрезка градиентов)

Суть: Ограничивает максимальное значение градиентов, предотвращая "взрыв" градиентов.

Стабилизирует обучение, особенно на длинных текстах.

In [ ]:
result_23 = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=2,
    gradient_clip_norm=0.5, model_key='23. Gradient Clipping',
)
technique_results['23. Gradient Clipping'] = result_23['val_macro_f1']

## Архитектурные модификации

###  Custom Classifier Head (Добавление дополнительных слоёв поверх BERT)


Суть: Вместо одного классификационного слоя добавляем несколько слоёв с dropout и активациями.

Даёт модели большеcapacity для сложных паттернов.

In [ ]:
result_24 = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=2,
    custom_model_class=CustomHeadModel, custom_model_kwargs={},
    model_key='24. Custom MLP Head',
)
technique_results['24. Custom MLP Head'] = result_24['val_macro_f1']

### Использование [CLS] + Mean Pooling

Суть: Вместо использования только [CLS] токена, усредняем все токены и конкатенируем с [CLS].

[CLS] может не захватывать всю информацию, особенно для длинных текстов.

In [ ]:
result_25 = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=2,
    custom_model_class=MeanPoolModel, custom_model_kwargs={},
    model_key='25. Mean Pooling + CLS',
)
technique_results['25. Mean Pooling + CLS'] = result_25['val_macro_f1']

## Работа с данными

### Active Learning (Активное обучение)

Суть: Модель сама выбирает, какие примеры ей нужны для обучения.

Как работает:
Обучаем модель на начальном датасете
Модель предсказывает на неразмеченных данных
Выбираешь примеры, где модель наименее уверена (entropy высокая)
Размечаешь их вручную или через аугментацию
Дообучаешь модел

In [ ]:
print("ACTIVE LEARNING")

# Шаг 1: Обучение на 50% данных
half = len(train_df) // 2
train_half = train_df.iloc[:half]

result_26_init = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=train_half, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=1, model_key='26. Active Learning (init)',
)

# Шаг 2: Поиск сложных примеров
model_26_init = result_26_init['model']
model_26_init.eval()

hard_texts, hard_labels = [], []
remaining = train_df.iloc[half:]

with torch.no_grad():
    for i in range(0, len(remaining), 16):
        batch = remaining.iloc[i:i+16]
        texts = batch[text_col].tolist()
        labels = batch['sentiment'].tolist()
        enc = result_26_init['tokenizer'](
            texts, padding=True, truncation=True, max_length=128, return_tensors='pt'
        ).to(DEVICE)
        probs = F.softmax(model_26_init(**enc).logits, dim=-1)
        entropy = -torch.sum(probs * torch.log(probs + 1e-10), dim=-1)
        top_k = max(1, int(len(texts) * 0.3))
        indices = torch.topk(entropy, min(top_k, len(entropy))).indices
        hard_texts.extend([texts[idx] for idx in indices.cpu()])
        hard_labels.extend([labels[idx] for idx in indices.cpu()])

hard_df = pd.DataFrame({text_col: hard_texts, 'sentiment': hard_labels})
combined_df = pd.concat([train_half, hard_df], ignore_index=True)

# Шаг 3: Дообучение
result_26 = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=combined_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=1e-5, epochs=1, model_key='26. Active Learning',
)
technique_results['26. Active Learning'] = result_26['val_macro_f1']

### Сurriculum Learning (Обучение по учебному плану)

Суть: Начинаем обучение с простых примеров, постепенно переходя к сложным.


Сортируем датасет по "сложности" (длина текста, количество стоп-слов и т.д.)
Первые эпохи — только простые примеры
Последние эпохи — все примеры

In [ ]:
print("CURRICULUM LEARNING")

# Сортируем по длине текста
lengths = [len(t) for t in train_df[text_col].tolist()]
sorted_idx = np.argsort(lengths)
sorted_texts = [train_df[text_col].tolist()[i] for i in sorted_idx]
sorted_labels = [train_df['sentiment'].tolist()[i] for i in sorted_idx]

tokenizer_cur = AutoTokenizer.from_pretrained(BEST_MODEL_NAME)
model_cur = AutoModelForSequenceClassification.from_pretrained(BEST_MODEL_NAME, num_labels=3).to(DEVICE)
optimizer_cur = torch.optim.AdamW(model_cur.parameters(), lr=2e-5)

for epoch in range(2):
    progress = (epoch + 1) / 2
    cur_size = int(len(sorted_texts) * (0.3 + 0.7 * progress))

    dataset_cur = SentimentDataset(sorted_texts[:cur_size], sorted_labels[:cur_size], tokenizer_cur, 128)
    loader_cur = DataLoader(dataset_cur, batch_size=16, shuffle=True)

    total_steps = len(loader_cur)
    scheduler_cur = get_cosine_schedule_with_warmup(optimizer_cur, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

    model_cur.train()
    for batch in loader_cur:
        optimizer_cur.zero_grad()
        out = model_cur(input_ids=batch['input_ids'].to(DEVICE), attention_mask=batch['attention_mask'].to(DEVICE), labels=batch['labels'].to(DEVICE))
        out.loss.backward()
        optimizer_cur.step()
        scheduler_cur.step()

    val_f1, _ = evaluate_model(model_cur, DataLoader(SentimentDataset(val_df[text_col].tolist(), val_df['sentiment'].tolist(), tokenizer_cur, 128), batch_size=16), DEVICE)
    print(f"   Epoch {epoch+1}/2 | Samples: {cur_size} | Val F1: {val_f1:.4f}")

final_f1_27, _ = evaluate_model(model_cur, DataLoader(SentimentDataset(val_df[text_col].tolist(), val_df['sentiment'].tolist(), tokenizer_cur, 128), batch_size=16), DEVICE)
technique_results['27. Curriculum Learning'] = final_f1_27
print(f" Val Macro-F1: {final_f1_27:.4f}")

### Mixup для текстов

Суть: Создаём синтетические примеры путём интерполяции эмбеддингов двух разных текстов.

Модель видит больше разнообразных примеров и учится обобщать.

In [ ]:
result_28 = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=2,
    mixup_alpha=0.2, model_key='28. Mixup',
)
technique_results['28. Mixup'] = result_28['val_macro_f1']

## Постобработка и инференс

### Threshold Tuning (Настройка порогов)

Суть: Вместо использования argmax для предсказания, настраиваем пороги для каждого класса отдельно.

Если модель систематически недооценивает один класс, можно снизить порог для него.

In [ ]:
print("THRESHOLD TUNING")

result_29_init = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=2, model_key='29. Threshold Tuning (train)',
)

# Поиск оптимальных порогов на validation
model_29 = result_29_init['model']
tokenizer_29 = result_29_init['tokenizer']

val_dataset_29 = SentimentDataset(val_df[text_col].tolist(), val_df['sentiment'].tolist(), tokenizer_29, 128)
val_loader_29 = DataLoader(val_dataset_29, batch_size=16, shuffle=False)

model_29.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for batch in val_loader_29:
        out = model_29(input_ids=batch['input_ids'].to(DEVICE), attention_mask=batch['attention_mask'].to(DEVICE))
        all_probs.append(F.softmax(out.logits, dim=-1).cpu().numpy())
        all_labels.extend(batch['labels'].cpu().numpy())

all_probs = np.vstack(all_probs)
all_labels = np.array(all_labels)

best_thresholds = [0.5, 0.5, 0.5]
for c in range(3):
    best_f1_c, best_t = 0, 0.5
    for t in np.arange(0.3, 0.7, 0.01):
        preds = (all_probs[:, c] >= t).astype(int)
        true_c = (all_labels == c).astype(int)
        tp = np.sum((preds == 1) & (true_c == 1))
        fp = np.sum((preds == 1) & (true_c == 0))
        fn = np.sum((preds == 0) & (true_c == 1))
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1_c = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        if f1_c > best_f1_c:
            best_f1_c, best_t = f1_c, t
    best_thresholds[c] = best_t

# Оценка с порогами
final_preds = []
for prob in all_probs:
    adj = prob.copy()
    for c, t in enumerate(best_thresholds):
        if prob[c] < t:
            adj[c] = 0
    final_preds.append(np.argmax(adj) if adj.sum() > 0 else np.argmax(prob))

f1_29 = f1_score(all_labels, final_preds, average='macro', zero_division=0)
technique_results['29. Threshold Tuning'] = f1_29
print(f" Val Macro-F1: {f1_29:.4f} | Пороги: {[round(t, 2) for t in best_thresholds]}")

### Ensemble with Different Seeds (Ансамбль с разными seed)

Суть: Обучаем одну и ту же модель 3-5 раз с разными random seed и усредняем предсказания.

Разные seed дают разные локальные минимумы, ансамбль снижает variance.

In [ ]:
print("ENSEMBLE WITH SEEDS")

seeds = [42, 1337, 2024]
models_30 = []

for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    result = train_transformer(
        model_name=BEST_MODEL_NAME, train_df=train_df, val_df=val_df,
        text_col=text_col, max_length=128, batch_size=16,
        learning_rate=2e-5, epochs=1, model_key=f'30. Seed={seed}',
    )
    models_30.append(result)

# Ансамбль
all_probs_list, all_labels_30 = [], []
val_dataset_30 = SentimentDataset(val_df[text_col].tolist(), val_df['sentiment'].tolist(), models_30[0]['tokenizer'], 128)
val_loader_30 = DataLoader(val_dataset_30, batch_size=16, shuffle=False)

with torch.no_grad():
    for batch in val_loader_30:
        probs_batch = []
        for res in models_30:
            res['model'].eval()
            out = res['model'](input_ids=batch['input_ids'].to(DEVICE), attention_mask=batch['attention_mask'].to(DEVICE))
            probs_batch.append(F.softmax(out.logits, dim=-1).cpu().numpy())
        all_probs_list.append(np.mean(probs_batch, axis=0))
        all_labels_30.extend(batch['labels'].cpu().numpy())

final_preds_30 = np.argmax(np.vstack(all_probs_list), axis=1)
f1_30 = f1_score(all_labels_30, final_preds_30, average='macro', zero_division=0)
technique_results['30. Ensemble with Seeds'] = f1_30
print(f" Val Macro-F1: {f1_30:.4f}")

## Дополнительные подходы

### Knowledge Distillation (Дистилляция знаний)

Суть: Обучаем маленькую модель (student) предсказывать то же, что и большая (teacher).

Student наследует знания teacher, но работает быстрее.

In [ ]:
print("KNOWLEDGE DISTILLATION")

print(" Пропускаем (требует много памяти)")
technique_results['31. Knowledge Distillation'] = baseline_final_f1
print(f" Val Macro-F1: {baseline_final_f1:.4f} (пропущено)")

### Multi-Task Learning (Многозадачное обучение)

Суть: Обучаем модель одновременно на нескольких задачах (например, тональность + длина текста + категория товара).

Модель учится более общим представлениям текста.

In [ ]:
print("MULTI-TASK LEARNING")

def get_aux_label(text):
    l = len(str(text))
    if l < 100: return 0
    elif l < 200: return 1
    elif l < 300: return 2
    elif l < 400: return 3
    return 4

mt_train = MTDataset(
    train_df[text_col].tolist(),
    train_df['sentiment'].tolist(),
    [get_aux_label(t) for t in train_df[text_col].tolist()],
    AutoTokenizer.from_pretrained(BEST_MODEL_NAME),
)
mt_loader = DataLoader(mt_train, batch_size=16, shuffle=True)

model_32 = MultiTaskModel(BEST_MODEL_NAME).to(DEVICE)
optimizer_32 = torch.optim.AdamW(model_32.parameters(), lr=2e-5)
total_steps = len(mt_loader) * 2
scheduler_32 = get_cosine_schedule_with_warmup(optimizer_32, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

for epoch in range(2):
    model_32.train()
    for batch in mt_loader:
        optimizer_32.zero_grad()
        out = model_32(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE),
            sent_labels=batch['sent_labels'].to(DEVICE),
            aux_labels=batch['aux_labels'].to(DEVICE),
        )
        if out.loss is not None:
            out.loss.backward()
        optimizer_32.step()
        scheduler_32.step()

val_dataset_32 = SentimentDataset(val_df[text_col].tolist(), val_df['sentiment'].tolist(), AutoTokenizer.from_pretrained(BEST_MODEL_NAME), 128)
val_loader_32 = DataLoader(val_dataset_32, batch_size=16, shuffle=False)
f1_32, _ = evaluate_model(model_32, val_loader_32, DEVICE)
technique_results['32. Multi-Task Learning'] = f1_32
print(f" Val Macro-F1: {f1_32:.4f}")

### Adversarial Training (Состязательное обучение)

Суть: Добавляем небольшой шум к эмбеддингам, чтобы модель была устойчивее.

Модель учится не полагаться на хрупкие паттерны

In [ ]:
result_33 = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=2,
    use_fgm=True, fgm_epsilon=1.0, model_key='33. Adversarial Training',
)
technique_results['33. Adversarial Training'] = result_33['val_macro_f1']

## Сравнение моделей из ячеек 15-27

In [ ]:
print("СРАВНЕНИЕ ТЕХНИК")

sorted_tech = sorted(technique_results.items(), key=lambda x: x[1], reverse=True)

print(f"\n{'Техника':<35} | {'Val Macro-F1':<12} | {'Прирост':<10}")
print("-" * 65)
for name, f1 in sorted_tech:
    print(f"{name:<35} | {f1:<12.4f} | {f1 - baseline_final_f1:+.4f}")

best_tech_name, best_tech_f1 = sorted_tech[0]
print(f"\n ЛУЧШАЯ ТЕХНИКА: {best_tech_name} (Val F1: {best_tech_f1:.4f})")

# Визуализация
plt.figure(figsize=(14, 8))
names = [x[0] for x in sorted_tech]
values = [x[1] for x in sorted_tech]
colors = ['#2ecc71' if v > baseline_final_f1 else '#e74c3c' for v in values]

plt.barh(names, values, color=colors)
plt.axvline(x=baseline_final_f1, color='blue', linestyle='--', linewidth=2, label=f'Baseline: {baseline_final_f1:.4f}')
plt.xlabel('Val Macro-F1', fontsize=12)
plt.title('Сравнение техник докрутки (Ячейки 21-33)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.legend()
for i, v in enumerate(values):
    plt.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

### Ансамль vs ЛУЧШАЯ ТЕХНИКА

In [ ]:
print("ФИНАЛЬНОЕ СРАВНЕНИЕ")

print(f"\n Базовая модель + аугментация + ансамбль: {baseline_final_f1:.4f}")
print(f" Лучшая техника докрутки: {best_tech_f1:.4f}")
print(f" Разница: {best_tech_f1 - baseline_final_f1:+.4f}")

diff = best_tech_f1 - baseline_final_f1
print("-" * 80)
if diff > 0.02:
    print(f" ОТЛИЧНЫЙ РЕЗУЛЬТАТ! Прирост +{diff:.4f}")
elif diff > 0.01:
    print(f" ХОРОШИЙ РЕЗУЛЬТАТ! Прирост +{diff:.4f}")
elif diff > 0:
    print(f" Небольшой прирост: +{diff:.4f}")
else:
    print(f"ℹ Базовая модель оказалась лучше ({diff:+.4f})")

# Финальный график
plt.figure(figsize=(10, 6))
categories = ['Ячейка 20\n(Базовая + Аугментация)', f'Ячейка 34\n({best_tech_name})']
vals = [baseline_final_f1, best_tech_f1]
colors_bar = ["#3498db", "#2ecc71"]

bars = plt.bar(categories, vals, color=colors_bar, width=0.6)
plt.ylabel("Val Macro-F1", fontsize=12)
plt.title("Финальное сравнение: Ячейка 20 vs Ячейка 34", fontsize=14, fontweight="bold")
plt.ylim(min(vals) - 0.05, max(vals) + 0.05)

for bar, val in zip(bars, vals):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
             f"{val:.4f}", ha="center", fontweight="bold", fontsize=12)
plt.tight_layout()
plt.show()

print(f"   Лучший Val Macro-F1: {max(baseline_final_f1, best_tech_f1):.4f}")